# RecyMind – Semaine 3 : modèle final et choix du seuil de confiance
Personne 2 · Réentraînement sur le dataset v2 de Personne 1, puis analyse du seuil sur les photos de l'équipe.

**Dossiers attendus sur Drive :** `recymind/data_v3/train/`, `val/` et `test_equipe/` (un sous-dossier par classe). Le lot `test_equipe` ne sert **jamais** à entraîner.
Règle du contrat : seuil initial 0.40 ; `tapis_vide` ne déclenche aucun tri ; confiance strictement inférieure au seuil = contrôle manuel.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tensorflow as tf, numpy as np, pandas as pd, json, time
import matplotlib.pyplot as plt

MODELE = "mobilenetv3"     # <-- mettre le gagnant de la semaine 2 : "mobilenetv3" ou "efficientnetb0"
BASE = "/content/drive/MyDrive/recymind/data_v3"
OUT = "/content/drive/MyDrive/recymind/modeles"
IMG, BATCH = (224, 224), 32

def charger(d, shuffle):
    return tf.keras.utils.image_dataset_from_directory(f"{BASE}/{d}", image_size=IMG, batch_size=BATCH, shuffle=shuffle, seed=42)

train_ds, val_ds, test_ds = charger("train", True), charger("val", False), charger("test_equipe", False)
class_names = train_ds.class_names
print(class_names)
assert "tapis_vide" in class_names and len(class_names) == 7
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

## 1. Entraînement final (tête, puis fine-tuning)

In [ ]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"), tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomBrightness(0.2, value_range=(0, 255)), tf.keras.layers.RandomContrast(0.2)])

if MODELE == "mobilenetv3":
    base = tf.keras.applications.MobileNetV3Small(input_shape=IMG+(3,), include_top=False, weights="imagenet")
else:
    base = tf.keras.applications.EfficientNetB0(input_shape=IMG+(3,), include_top=False, weights="imagenet")
base.trainable = False
x_in = tf.keras.Input(IMG + (3,))
x = base(augment(x_in), training=False)
x = tf.keras.layers.Dropout(0.3)(tf.keras.layers.GlobalAveragePooling2D()(x))
model = tf.keras.Model(x_in, tf.keras.layers.Dense(len(class_names), activation="softmax")(x))

stop = tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor="val_accuracy")
model.compile(tf.keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", ["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=[stop])
base.trainable = True
for c in base.layers[:-30]: c.trainable = False
model.compile(tf.keras.optimizers.Adam(1e-5), "sparse_categorical_crossentropy", ["accuracy"])
model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[stop])

## 2. Probabilités sur les photos de l'équipe

In [ ]:
probs = model.predict(test_ds, verbose=0)
y_true = np.concatenate([y.numpy() for _, y in test_ds])
pred, conf = probs.argmax(1), probs.max(1)
i_vide = class_names.index("tapis_vide")
print("Accuracy test équipe :", round((pred == y_true).mean(), 3))

## 3. Analyse du seuil
On ne garde que les images que le backend essaierait de trier (classe prédite ≠ `tapis_vide`). Pour chaque seuil :
- **% au contrôle manuel** : part des déchets envoyés à l'opérateur (trop élevé = le système ne sert à rien) ;
- **accuracy des acceptés** : part de bons tris parmi les déchets triés automatiquement ;
- **erreurs acceptées** : mauvais tris qui partent dans un bac sans contrôle (à minimiser).

In [ ]:
dechet = pred != i_vide
p, c, y = pred[dechet], conf[dechet], y_true[dechet]
lignes = []
for t in np.arange(0.10, 0.96, 0.05):
    acceptes = c >= t                          # règle du contrat : >= seuil accepté, < seuil contrôle
    lignes.append({
        "seuil": round(t, 2),
        "pct_controle_manuel": round(100 * (1 - acceptes.mean()), 1),
        "accuracy_acceptes": round(float((p == y)[acceptes].mean()), 3) if acceptes.any() else None,
        "pct_erreurs_acceptees": round(100 * float(((p != y) & acceptes).mean()), 1),
    })
tab = pd.DataFrame(lignes)
tab.to_csv(f"{OUT}/analyse_seuil.csv", index=False)
tab

In [ ]:
fig, ax = plt.subplots()
ax.plot(tab.seuil, tab.pct_controle_manuel, label="% au contrôle manuel")
ax.plot(tab.seuil, tab.pct_erreurs_acceptees, label="% d'erreurs acceptées")
ax.axvline(0.40, ls="--", c="gray", label="seuil actuel (0.40)")
ax.set_xlabel("seuil de confiance"); ax.set_ylabel("%"); ax.legend(); plt.show()

In [ ]:
# Critère proposé : accuracy des acceptés >= 95 % avec au plus 30 % de contrôle manuel (à discuter en équipe)
ok = tab[(tab.accuracy_acceptes >= 0.95) & (tab.pct_controle_manuel <= 30)]
print("Seuil recommandé :", None if ok.empty else float(ok.seuil.iloc[0]))
print("Si aucun seuil ne convient : garder 0.40 et améliorer les données (photos des classes confondues).")

## 4. Évaluation finale, sauvegarde

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
print(classification_report(y_true, pred, target_names=class_names))
ConfusionMatrixDisplay(confusion_matrix(y_true, pred), display_labels=class_names).plot(xticks_rotation=45)
plt.title("Matrice de confusion – photos de l'équipe"); plt.show()
print("Déchets pris pour un tapis vide (jamais triés) :", int(((pred == i_vide) & (y_true != i_vide)).sum()))

img = np.random.randint(0, 255, (1,)+IMG+(3,)).astype("float32"); model.predict(img, verbose=0)
t0 = time.time(); [model.predict(img, verbose=0) for _ in range(50)]
print("Temps d'inférence moyen (ms) :", round((time.time() - t0) / 50 * 1000, 1))

!mkdir -p "{OUT}"
model.save(f"{OUT}/modele_final.keras")
json.dump(class_names, open(f"{OUT}/classes_final.json", "w"))